In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

In [2]:
data=pd.read_csv('football_qa_dataset_full.csv')

In [3]:
#same train / validation / test split as rnn.ipynb so the results are comparable
train_val_data=data.sample(frac=0.8,random_state=42)
test_data=data.drop(train_val_data.index)

val_data=train_val_data.sample(frac=0.15,random_state=42)
train_data=train_val_data.drop(val_data.index)

print(len(train_data),len(val_data),len(test_data))

1960 346 576


In [4]:
#pretrained sentence embedding model (downloads ~400MB the first time)
#it already knows that e.g. "won" and "was crowned champion" mean similar things
encoder=SentenceTransformer('all-mpnet-base-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
def embed(questions):
    return encoder.encode(list(questions),normalize_embeddings=True,batch_size=64)

train_X=embed(train_data['question'])
val_X=embed(val_data['question'])
test_X=embed(test_data['question'])

print(train_X.shape)

(1960, 768)


In [6]:
#pick the regularisation strength (C) using the validation set
best_val_acc=0
best_model=None

for C in [1,10,100]:
    model=LogisticRegression(C=C,max_iter=3000)
    model.fit(train_X,train_data['answer'])
    val_acc=model.score(val_X,val_data['answer'])
    print(f'C={C},val acc:{val_acc:.3f}')

    if val_acc>best_val_acc:
        best_val_acc=val_acc
        best_model=model

model=best_model

C=1,val acc:0.182


C=10,val acc:0.234


C=100,val acc:0.269


In [7]:
print(f'train acc:{model.score(train_X,train_data["answer"]):.3f}')
print(f'val acc:{model.score(val_X,val_data["answer"]):.3f}')
print(f'test acc:{model.score(test_X,test_data["answer"]):.3f}')

train acc:0.973
val acc:0.269
test acc:0.205


In [8]:
def predict(model,question,threshold=0.1):
    proba=model.predict_proba(embed([question]))[0]
    index=np.argmax(proba)

    if proba[index] < threshold:
        print('idk')
    else:
        print(model.classes_[index])

In [9]:
predict(model,"team lost the final of 1954 FIFA World Cup?")

Hungary


In [10]:
predict(model,"Who won the 1954 FIFA World Cup?")

WestGermany


In [11]:
predict(model,"Which MLS club did Lionel Messi join in 2023?")

Inter-Miami